In [6]:
import re
import pandas as pd

# --- CONFIG ---
csv_files = ['2011 RVI Website Data_07082026.csv','2016 RVI Website Data_07082026.csv','2021 RVI Website Data_07082026.csv' ]
id_column = "sa2_code"  # your join key

# --- MERGE (loop; no functools) ---
merged = pd.read_csv("input/" + csv_files[0]).rename(columns=str.strip)
for f in csv_files[1:]:
    df = pd.read_csv("input/" + f).rename(columns=str.strip)
    merged = pd.merge(merged, df, on=id_column, how="outer")  # will create _x/_y suffixes

# --- HELPER: strip pandas merge suffixes (including chained ones) ---
_suffix_re = re.compile(r'(?:(_x|_y)|(\.\d+))+$')  # handles _x, _y, .1, .2 and repeated chains

def base_name(col: str) -> str:
    return _suffix_re.sub('', col)

# --- GROUP BY BASE NAME, FILL PRIMARY, DROP EXTRAS ---
cols = list(merged.columns)
by_base = {}
for c in cols:
    if c == id_column:
        continue
    b = base_name(c)
    by_base.setdefault(b, []).append(c)

to_drop = []
for b, col_list in by_base.items():
    if len(col_list) == 1:
        continue  # nothing to merge for this base name

    # Keep deterministic order: as they appear in the dataframe (left→right)
    # Build a block: [primary, alt1, alt2, ...]
    block = merged[col_list]

    # Fill primary ONLY where it's missing, using later columns’ values
    primary = col_list[0]
    filled_primary = block.bfill(axis=1).iloc[:, 0]

    # Assign back to the primary
    merged[primary] = filled_primary

    # Schedule all the other columns for deletion
    to_drop.extend(col_list[1:])

# Drop duplicates after filling
if to_drop:
    merged.drop(columns=to_drop, inplace=True)

# --- CLEAN UP COLUMN NAMES ---
merged.rename(columns={c: base_name(c) for c in merged.columns}, inplace=True)

# --- OUTPUT ---
merged.to_csv("input/combined.csv", index=False)
print("✅ Done: filled primary columns from suffixed duplicates and removed extras → merged_output.csv")

✅ Done: filled primary columns from suffixed duplicates and removed extras → merged_output.csv


In [8]:
import json
from pathlib import Path
from collections import Counter
import ast 

import pandas as pd

# -------------------
# CONFIG
# -------------------
INPUT_CSV = "input/combined.csv"       
OUTPUT_JSON = "output/state_year_cards.json"
YEARS = [2011, 2016, 2021]             # include 2011 if needed, e.g. [2011, 2016, 2021]
STATE_COL = "state"              # e.g., NSW, QLD, etc.
MEDIAN_SERIES_COL = "ts_rent"   # e.g. "[120, 130, 140]"
MEDIAN_LABELS_COL = "ts_rent_labels"   # e.g. "[2011, 2016, 2021]"

# If your input uses full state names or mixed case, normalise here:
STATE_ALIASES = {
    "new south wales": "nsw", "nsw": "nsw",
    "queensland": "qld", "qld": "qld",
    "victoria": "vic", "vic": "vic",
    "south australia": "sa", "sa": "sa",
    "western australia": "wa", "wa": "wa",
    "tasmania": "tas", "tas": "tas",
    "northern territory": "nt", "nt": "nt",
    "australian capital territory": "act", "act": "act",
}

# Optional label mapping for lcv categories (uncomment & edit if your lcv_* are codes)
# lcv_LABELS = { "1": "Very Low", "2": "Low", "3": "Moderate", "4": "High", "5": "Very High" }


# -------------------
# Helpers
# -------------------
def norm_state(s: str) -> str:
    if pd.isna(s):
        return ""
    key = str(s).strip().lower()
    return STATE_ALIASES.get(key, key)

def fmt_count(n: float) -> str:
    try:
        n = float(n or 0)
    except Exception:
        return "0"
    if n >= 1_000_000:
        # 2dp then strip trailing zeros
        txt = f"{n/1_000_000:.2f}".rstrip("0").rstrip(".")
        return f"{txt}m"
    if n >= 1_000:
        return f"{n/1_000:.0f}k"
    return f"{int(round(n))}"

def fmt_pct(num: float, den: float) -> str:
    try:
        num = float(num or 0); den = float(den or 0)
    except Exception:
        return ""
    if den <= 0:
        return ""
    p = (num / den) * 100
    # 1dp, strip .0
    return f"{p:.1f}%".rstrip("0").rstrip(".")

def combo(n: float, d: float) -> str:
    """Return 'X (Y%)' with X formatted count and Y% over denominator d."""
    if (d or 0) <= 0:
        return fmt_count(n)
    return f"{fmt_count(n)} ({fmt_pct(n, d)})"

def get_col(year, base):
    return f"{base}_{year}"

def mode_categorical(series: pd.Series) -> str:
    """Most common non-null, non-empty. Tie-break: numeric sort if possible else lexicographic."""
    s = series.dropna().astype(str).str.strip()
    s = s[s != ""]
    if s.empty:
        return ""
    counts = Counter(s)
    max_count = max(counts.values())
    candidates = [v for v, c in counts.items() if c == max_count]
    # numeric sort if possible for stable tie-breaking
    try:
        candidates_sorted = sorted(candidates, key=lambda x: (float(x), str(x)))
    except Exception:
        candidates_sorted = sorted(candidates)
    return candidates_sorted[0]

def median_rent_value(state_key: str, year: int) -> str:
    """
    Calculate the median rent (across all SA2s) for a given state and year.
    Returns a formatted string like '480 pw' or '--' if no data.
    """
    col = f"{str(year)[-2:]}_m_rent_common"  # predictable column name
    vals = pd.to_numeric(df.loc[df[STATE_COL] == state_key, col], errors="coerce").dropna()

    if vals.empty:
        return "--"

    v = vals.median()
    return f"$ {int(round(v))} pw"

# -------------------
# Load & normalise
# -------------------
df = pd.read_csv(INPUT_CSV)
df[STATE_COL] = df[STATE_COL].map(norm_state)

# -------------------
# Pre-compute lcv mode by state×year
# -------------------
lcv_mode_by_state_year = {}
for year in YEARS:
    lcv_col = f"lcv_{year}"
    if lcv_col in df.columns:
        tmp = df[[STATE_COL, lcv_col]].copy()
        tmp[STATE_COL] = tmp[STATE_COL].map(norm_state)
        modes = tmp.groupby(STATE_COL, dropna=False)[lcv_col].apply(mode_categorical)
        for sk, v in modes.items():
            if sk:
                # If you have code -> label mapping, apply here:
                # v = lcv_LABELS.get(str(v), str(v))
                lcv_mode_by_state_year[(sk, year)] = str(v)
    else:
        # column missing: leave as empty string for that year
        pass



# -------------------
# Parse list-valued cells and precompute median rent trend by state
# -------------------
import ast

def parse_list_cell(val):
    """Convert a string like '[120,130,140]' into a Python list."""
    if pd.isna(val):
        return []
    try:
        out = ast.literal_eval(str(val))
        if isinstance(out, (list, tuple)):
            return list(out)
        return []
    except Exception:
        return []


def median_rent_trend(df, state_key: str, year: int):

    values_col = f"ts_rent_{year}"
    labels_col = f"ts_rent_labels_{year}"  # change if needed

    print(values_col)
    
    if values_col not in df.columns or labels_col not in df.columns:
       
        return [], []
        

    subset = df.loc[df[STATE_COL] == state_key]
    
    

    all_series = []
    labels = None

    for _, row in subset[[values_col, labels_col]].iterrows():
        vals = parse_list_cell(row[values_col])
        labs = parse_list_cell(row[labels_col])

        if not vals or not labs:
            continue

        if labels is None:
            labels = labs

        if len(vals) != len(labels):
            continue

        all_series.append(vals)

    if not all_series or labels is None:
        return [], []

    arr = pd.DataFrame(all_series).astype(float)
    medians = arr.median(axis=0).tolist()

    cleaned = [int(x) if str(x).isdigit() else x for x in labels]

    return medians, cleaned

# --- LANGUAGE TOTALS ACROSS STATES (for all languages in merged_output.csv) ---

LANG_COLS = [
    "spanish", "arabic", "hindi", "punjabi", "vietnamese", "japanese",
    "korean", "mandarin", "samoan", "tagalog", "all_other_lang"
]

LANG_LABELS = {
    "spanish": "Spanish",
    "arabic": "Arabic",
    "hindi": "Hindi",
    "punjabi": "Punjabi",
    "vietnamese": "Vietnamese",
    "japanese": "Japanese",
    "korean": "Korean",
    "mandarin": "Mandarin",
    "samoan": "Samoan",
    "tagalog": "Tagalog",
    "all_other_lang": "Other"
}

def total_languages(df, year):
    """Return total counts and labels for all languages for the given year."""
    totals = []
    labels = []
    for lang in LANG_COLS:
        col = f"{lang}_{year}"
        if col in df.columns:
            total = df[col].sum(skipna=True)
            totals.append(float(total))
            labels.append(LANG_LABELS.get(lang, lang.replace("_", " ").title()))
    return totals, labels

# TENURE

TENURE_COLS = [
    "own_occ",
    "rented",
    "other_tenure"
]

TENURE_LABELS = {
    "own_occ": "Owner Occ.",
    "rented": "Rented",
    "other_tenure": "Other"
}

def total_tenure(df, year):
    """
    Return total counts, plain labels, and chart_labels (with %).
    Example chart_labels: ["Own Occ (45%)", "Rented (50%)", ...]
    """
    totals = []
    labels = []
    chart_labels = []

    for tenure in TENURE_COLS:
        col = f"{tenure}_{year}"
        if col in df.columns:
            total = df[col].sum(skipna=True)
            totals.append(float(total))
            labels.append(TENURE_LABELS.get(tenure, tenure.title()))

    # Compute chart labels with percentages
    total_sum = sum(totals)
    for lbl, val in zip(labels, totals):
        if total_sum > 0:
            pct = (val / total_sum) * 100
            chart_labels.append(f"{pct:.1f}%")
        else:
            chart_labels.append(f"0%")

    return totals, labels, chart_labels


# -------------------
# Aggregate and build cards
# -------------------
records = []

for year in YEARS:
    cols_needed = [
        get_col(year, "total_persons"),
        get_col(year, "total_renters"),
        get_col(year, "recrenters"),
        get_col(year, "total_dwellings"),
        get_col(year, "own_occ"),
        get_col(year, "rented"),
        get_col(year, "other_tenure"),
        get_col(year, "tenure not stated"),
        get_col(year, "tenure n_a"),
        get_col(year, "rent_stress"),
        get_col(year, "young"),
        get_col(year, "older"),
        get_col(year, "unemployed"),
        get_col(year, "single_parent"),
        get_col(year, "low_ed"),
        get_col(year, "assist"),
        get_col(year, "indig"),
        get_col(year, "english"),
        get_col(year, "spanish"),
        get_col(year, "arabic"),
        get_col(year, "hindi"),
        get_col(year, "punjabi"),
        get_col(year, "vietnamese"),
        get_col(year, "japanese"),
        get_col(year, "korean"),
        get_col(year, "mandarin"),
        get_col(year, "samoan"),
        get_col(year, "tagalog"),
        get_col(year, "all_other_lang"),
        get_col(year, "public_community"),
        get_col(year, "boardinghouse"),
        get_col(year, "residential_park"),
        get_col(year, "rvi"),
        # lcv handled as mode; don't need to sum it
        # median rent columns are handled later via helper
    ]
    agg = df.groupby(STATE_COL, dropna=False).agg({c: "sum" for c in cols_needed if c in df.columns}).reset_index()
    agg = agg.rename(columns={STATE_COL: "state_key"})

    for _, row in agg.iterrows():
        state_key = row["state_key"]
        if not state_key:
            continue

        tp = row.get(get_col(year, "total_persons"), 0)
        tr = row.get(get_col(year, "total_renters"), 0)
        recent_renters = row.get(get_col(year, "recrenters"), 0) 
        td = row.get(get_col(year, "total_dwellings"), 0)

        own = row.get(get_col(year, "own_occ"), 0)
        rent_stress = row.get(get_col(year, "rent_stress"), 0)

        young = row.get(get_col(year, "young"), 0)
        older = row.get(get_col(year, "older"), 0)
        unemp = row.get(get_col(year, "unemployed"), 0)
        single_parent = row.get(get_col(year, "single_parent"), 0)
        low_ed = row.get(get_col(year, "low_ed"), 0)
        assist = row.get(get_col(year, "assist"), 0)
        indig = row.get(get_col(year, "indig"), 0)

        english = row.get(get_col(year, "english"), 0)
        lang_cols = [
            "spanish","arabic","hindi","punjabi","vietnamese","japanese",
            "korean","mandarin","samoan","tagalog","all_other_lang"
        ]
        other_lang = sum(row.get(get_col(year, lc), 0) for lc in lang_cols)

        public_comm = row.get(get_col(year, "public_community"), 0)
        boarding = row.get(get_col(year, "boardinghouse"), 0)
        res_park = row.get(get_col(year, "residential_park"), 0)

        rvi = row.get(get_col(year, "rvi"), None)
        # lcv as most common category across SA2s in this state-year:
        lcv_mode = lcv_mode_by_state_year.get((state_key, year), "")

        med_rent = median_rent_value(state_key, year)
        trend_vals, trend_labels = median_rent_trend(df, state_key, year)
        print(trend_vals, trend_labels)

        
        # Example use:
        lang_totals, lang_labels = total_languages(df, year)

        tenure_totals, tenure_labels, tenure_chart_labels = total_tenure(df, year)

        pretty_state = state_key.upper()

        rows = [
            {"header": "Name", "state": state_key, "year": str(year), "value": pretty_state, "labels": "", "chart":""},
            {"header": "SA2 Code", "state": state_key, "year": str(year), "value": "--", "labels": "", "chart":""},
            {"header": "Rental Vulnerability Index", "state": state_key, "year": str(year), "value": "--", "labels": "", "chart":""},
            {"header": "Primary Vulnerability Category", "state": state_key, "year": str(year), "value": "--", "labels": "", "chart":""},
            {"header": "Secondary Vulnerability Category", "state": state_key, "year": str(year), "value": "--", "labels": "", "chart":""},
            {"header": "Tertiary Vulnerability Category", "state": state_key, "year": str(year), "value": "--", "labels": "", "chart":""},
            {"header": "Rent Stress", "state": state_key, "year": str(year), "value": combo(rent_stress, tr), "labels": "", "chart":""},
            {"header": "Number of Renters", "state": state_key, "year": str(year), "value": combo(tr, tp), "labels": "", "chart":""},
            {"header": "Recently Moved", "state": state_key, "year": str(year), "value": combo(recent_renters, tr), "labels": "", "chart":""}, 
            {"header": "Bonds Held", "state": state_key, "year": str(year), "value": "--", "labels": "", "chart":""},
            {"header": "Median Rent", "state": state_key, "year": str(year), "value": med_rent, "labels": "", "chart":""},
            {"header": "Affordable Rentals", "state": state_key, "year": str(year), "value": "--", "labels": "", "chart":""},
            {"header": "Bonds Held (Trend)", "state": state_key, "year": str(year), "value": "--", "labels": "", "chart":""},
            {"header": "Median Rent (Trend)", "state": state_key, "year": str(year), "value": trend_vals, "labels": trend_labels, "chart":"line"},
            {"header": "Affordable Rentals (Trend)", "state": state_key, "year": str(year), "value": "--", "labels": "--", "chart":""},
            {"header": "Public/Community Housing", "state": state_key, "year": str(year), "value": combo(public_comm, td), "labels": "", "chart":""},
            {"header": "Boarding Houses", "state": state_key, "year": str(year), "value": combo(boarding, td), "labels": "", "chart":""},
            {"header": "Residential Parks", "state": state_key, "year": str(year), "value": combo(res_park, td), "labels": "", "chart":""},
            {"header": "Home Ownership", "state": state_key, "year": str(year), "value": tenure_totals, "labels": tenure_labels, "chart":"doughnut", "chart_labels": tenure_chart_labels},
            {"header": "Younger", "state": state_key, "year": str(year), "value": combo(young, tr), "labels": "", "chart":""},
            {"header": "Older", "state": state_key, "year": str(year), "value": combo(older, tr), "labels": "", "chart":""},
            {"header": "Unemployed", "state": state_key, "year": str(year), "value": combo(unemp, tr), "labels": "", "chart":""},
            {"header": "Single Parent", "state": state_key, "year": str(year), "value": combo(single_parent, tr), "labels": "", "chart":""},
            {"header": "Lower Education Level", "state": state_key, "year": str(year), "value": combo(low_ed, tr), "labels": "", "chart":""},
            {"header": "Disabled", "state": state_key, "year": str(year), "value": combo(assist, tr), "labels": "", "chart":""},
            {"header": "Indigenous", "state": state_key, "year": str(year), "value": combo(indig, tr), "labels": "", "chart":""},
            {"header": "English Speakers", "state": state_key, "year": str(year), "value": combo(english, tr), "labels": "", "chart":""},
            {"header": "Other Languages", "state": state_key, "year": str(year), "value": lang_totals, "labels": lang_labels, "chart":"doughnut"},
        ]
        records.extend(rows)

# -------------------
# Save (compact: one JSON per line)
# -------------------
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    f.write("[\n")
    for i, rec in enumerate(records):
        json.dump(rec, f, ensure_ascii=False)
        # comma after each row except the last
        if i < len(records) - 1:
            f.write(",\n")
        else:
            f.write("\n")
    f.write("]\n")

print(f"Wrote {len(records)} rows (comma-separated, newline-delimited JSON array) to {OUTPUT_JSON}")

ts_rent_2011
[157.0, 250.0, 370.0] [2001, 2006, 2011]
ts_rent_2011
[150.0, 195.0, 270.0] [2001, 2006, 2011]
ts_rent_2011
[130.0, 155.0, 288.0] [2001, 2006, 2011]
ts_rent_2011
[100.0, 118.0, 86.0] [2001, 2006, 2011]
ts_rent_2011
[146.0, 200.0, 300.0] [2001, 2006, 2011]
ts_rent_2011
[110.0, 152.0, 220.0] [2001, 2006, 2011]
ts_rent_2011
[100.0, 135.0, 200.0] [2001, 2006, 2011]
ts_rent_2011
[145.0, 180.0, 270.0] [2001, 2006, 2011]
ts_rent_2011
[125.0, 165.0, 290.0] [2001, 2006, 2011]
ts_rent_2016
[242.5, 354.0, 365.5] [2006, 2011, 2016]
ts_rent_2016
[200.0, 275.0, 350.0] [2006, 2011, 2016]
ts_rent_2016
[155.0, 280.0, 350.0] [2006, 2011, 2016]
ts_rent_2016
[92.0, 70.0, 134.0] [2006, 2011, 2016]
ts_rent_2016
[200.0, 300.0, 340.0] [2006, 2011, 2016]
ts_rent_2016
[155.0, 220.0, 265.0] [2006, 2011, 2016]
ts_rent_2016
[130.0, 200.0, 233.0] [2006, 2011, 2016]
ts_rent_2016
[185.0, 275.0, 321.0] [2006, 2011, 2016]
ts_rent_2016
[160.0, 290.0, 340.0] [2006, 2011, 2016]
ts_rent_2021
[360.0, 370.0, 430